In [ ]:
# ============================================================

# ============================================================

# STEP 1: INSTALL & IMPORT (FIXED)
print("="*80)
print("STEP 1: Installing packages...")
print("="*80 + "\n")

!pip install openpyxl reportlab catboost xgboost scikit-learn seaborn scipy -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import gc
import os
from datetime import datetime
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from catboost import CatBoostClassifier
import xgboost as xgb

# FIX: mcnemar is in statsmodels, not scipy
!pip install statsmodels -q
from statsmodels.stats.contingency_tables import mcnemar

print("✅ All libraries imported\n")

# STEP 2: MOUNT DRIVE
print("="*80)
print("STEP 2: Mounting Google Drive...")
print("="*80 + "\n")

from google.colab import drive
drive.mount('/content/drive')

# PATHS
BASE_DATA_PATH = "/content/drive/MyDrive/Kaggle Dataset"
BASE_RESULTS_PATH = f"{BASE_DATA_PATH}/Results/ResultsAfterOptimization"
STATS_PATH = f"{BASE_RESULTS_PATH}/Statistical_Analysis"

os.makedirs(STATS_PATH, exist_ok=True)

print(f"✅ Drive mounted")
print(f"📁 Dataset: {BASE_DATA_PATH}")
print(f"📁 Results: {STATS_PATH}\n")

# Verify paths
if os.path.exists(f"{BASE_DATA_PATH}/Relax"):
    print("✅ Relax folder found")
else:
    print("❌ ERROR: Relax folder NOT FOUND!")
    print("   Your actual path might be different. Check Google Drive.")
    raise FileNotFoundError("Relax folder not found")

if os.path.exists(f"{BASE_DATA_PATH}/Stroop"):
    print("✅ Stroop folder found\n")
else:
    print("❌ ERROR: Stroop folder NOT FOUND!")
    raise FileNotFoundError("Stroop folder not found")

# STEP 3: DEFINE FUNCTIONS
print("="*80)
print("STEP 3: Defining helper functions...")
print("="*80 + "\n")

def load_grouped(folder, label):
    files = sorted(glob.glob(f"{folder}/*.csv"))
    dfs, groups = [], []
    for i, f in enumerate(files):
        df = pd.read_csv(f)
        if df.shape[1] > 1: df = df.iloc[:, 1:]
        dfs.append(df)
        groups.append(np.full(len(df), f"label{label}_sub{i}"))
    data = pd.concat(dfs, ignore_index=True)
    return data.values.astype(np.float32), np.full(len(data), label), np.hstack(groups)

def load_with_subjects(folder, label):
    files = sorted(glob.glob(f"{folder}/*.csv"))
    dfs, subject_ids = [], []
    for i, f in enumerate(files):
        df = pd.read_csv(f)
        if df.shape[1] > 1: df = df.iloc[:, 1:]
        dfs.append(df)
        subject_ids.append(np.full(len(df), i))
    data = pd.concat(dfs, ignore_index=True)
    return data.values.astype(np.float32), np.full(len(data), label), np.hstack(subject_ids)

def cohens_d(acc1, acc2):
    pooled_std = np.sqrt((np.var(acc1) + np.var(acc2)) / 2)
    return (np.mean(acc1) - np.mean(acc2)) / pooled_std if pooled_std > 0 else 0

def robust_bootstrap_ci(accuracies, n_bootstrap=10000, ci=0.95):
    bootstrap_means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(accuracies, size=len(accuracies), replace=True)
        bootstrap_means.append(np.mean(sample))
    alpha = 1 - ci
    lower = np.percentile(bootstrap_means, alpha/2 * 100)
    upper = np.percentile(bootstrap_means, (1 - alpha/2) * 100)
    return lower, upper

print("✅ Functions defined\n")

# STEP 4: LOAD DATA
print("="*80)
print("STEP 4: Loading EEG data...")
print("="*80 + "\n")

X1, y1, g1 = load_grouped(f"{BASE_DATA_PATH}/Relax", 0)
X2, y2, g2 = load_grouped(f"{BASE_DATA_PATH}/Stroop", 1)
X = np.vstack((X1, X2))
y = np.hstack((y1, y2))
groups = np.hstack((g1, g2))

X_scaled = StandardScaler().fit_transform(X)
print(f"✅ Loaded {len(X):,} samples, {X.shape[1]} features\n")

# STEP 5: DEFINE MODELS
print("="*80)
print("STEP 5: Defining ML models...")
print("="*80 + "\n")

models_dict = {
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=2),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=2),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42, n_jobs=2, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(iterations=100, depth=6, random_state=42, verbose=False, thread_count=2),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42)
}

si_models = {
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, depth=6, random_state=42, verbose=False, thread_count=2),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42, n_jobs=2, eval_metric='logloss')
}

print("✅ Models defined\n")

# ============================================================
# GAP A: CORRECTED STATISTICAL TESTS + EFFECT SIZES
# ============================================================

print("\n" + "="*80)
print("GAP A: CORRECTED STATISTICAL TESTS + EFFECT SIZES")
print("Estimated time: 30 minutes")
print("="*80 + "\n")

# Subject-Dependent Training
print("Training Subject-Dependent models (5 models)...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sd_predictions = {}

for name, model in models_dict.items():
    print(f"  {name}...", end=" ")
    preds = np.zeros(len(y))
    fold_accs = []

    for train_idx, test_idx in skf.split(X_scaled, y):
        model_clone = type(model)(**model.get_params())
        model_clone.fit(X_scaled[train_idx], y[train_idx])
        preds[test_idx] = model_clone.predict(X_scaled[test_idx])
        fold_accs.append(accuracy_score(y[test_idx], preds[test_idx]))

    sd_predictions[name] = {'preds': preds, 'fold_accs': fold_accs, 'mean': np.mean(fold_accs)}
    print(f"Done (Acc: {np.mean(fold_accs)*100:.2f}%)")

# Subject-Independent Training
print("\nTraining Subject-Independent models (3 models)...")
gkf = GroupKFold(n_splits=5)
si_predictions = {}

for name, model in si_models.items():
    print(f"  {name}...", end=" ")
    preds = np.zeros(len(y))
    fold_accs = []

    for train_idx, test_idx in gkf.split(X_scaled, y, groups):
        model_clone = type(model)(**model.get_params())
        model_clone.fit(X_scaled[train_idx], y[train_idx])
        preds[test_idx] = model_clone.predict(X_scaled[test_idx])
        fold_accs.append(accuracy_score(y[test_idx], preds[test_idx]))

    si_predictions[name] = {'preds': preds, 'fold_accs': fold_accs, 'mean': np.mean(fold_accs)}
    print(f"Done (Acc: {np.mean(fold_accs)*100:.2f}%)")

# Pairwise Statistical Tests
print("\nPerforming pairwise statistical tests with Bonferroni correction...")
model_names = list(models_dict.keys())
all_results = []
n_comparisons = 0

# SD comparisons
for i, model1 in enumerate(model_names):
    for j, model2 in enumerate(model_names):
        if i < j:
            n_comparisons += 1

            preds1 = sd_predictions[model1]['preds']
            preds2 = sd_predictions[model2]['preds']

            correct1 = (preds1 == y)
            correct2 = (preds2 == y)

            contingency = np.array([
                [np.sum(correct1 & correct2), np.sum(correct1 & ~correct2)],
                [np.sum(~correct1 & correct2), np.sum(~correct1 & ~correct2)]
            ])

            # FIX: statsmodels mcnemar returns different structure
            result = mcnemar(contingency, exact=False, correction=True)

            fold_accs1 = sd_predictions[model1]['fold_accs']
            fold_accs2 = sd_predictions[model2]['fold_accs']
            effect_size = cohens_d(fold_accs1, fold_accs2)

            alpha = 0.05

            all_results.append({
                'Comparison': f'{model1} vs {model2}',
                'Validation': 'Subject-Dependent',
                'Model1_Acc_%': sd_predictions[model1]['mean'] * 100,
                'Model2_Acc_%': sd_predictions[model2]['mean'] * 100,
                'Diff_pp': (sd_predictions[model1]['mean'] - sd_predictions[model2]['mean']) * 100,
                'p_value_raw': result.pvalue,
                'p_value_corrected': min(result.pvalue * n_comparisons, 1.0),
                'Significant_after_correction': result.pvalue * n_comparisons < alpha,
                'Cohens_d': effect_size,
                'Effect_Size': 'Large' if abs(effect_size) > 0.8 else 'Medium' if abs(effect_size) > 0.5 else 'Small'
            })

# SI comparisons
si_names = list(si_models.keys())
n_si_comparisons = 3

for i, model1 in enumerate(si_names):
    for j, model2 in enumerate(si_names):
        if i < j:
            preds1 = si_predictions[model1]['preds']
            preds2 = si_predictions[model2]['preds']

            correct1 = (preds1 == y)
            correct2 = (preds2 == y)

            contingency = np.array([
                [np.sum(correct1 & correct2), np.sum(correct1 & ~correct2)],
                [np.sum(~correct1 & correct2), np.sum(~correct1 & ~correct2)]
            ])

            result = mcnemar(contingency, exact=False, correction=True)

            fold_accs1 = si_predictions[model1]['fold_accs']
            fold_accs2 = si_predictions[model2]['fold_accs']
            effect_size = cohens_d(fold_accs1, fold_accs2)

            all_results.append({
                'Comparison': f'{model1} vs {model2}',
                'Validation': 'Subject-Independent',
                'Model1_Acc_%': si_predictions[model1]['mean'] * 100,
                'Model2_Acc_%': si_predictions[model2]['mean'] * 100,
                'Diff_pp': (si_predictions[model1]['mean'] - si_predictions[model2]['mean']) * 100,
                'p_value_raw': result.pvalue,
                'p_value_corrected': min(result.pvalue * n_si_comparisons, 1.0),
                'Significant_after_correction': result.pvalue * n_si_comparisons < alpha,
                'Cohens_d': effect_size,
                'Effect_Size': 'Large' if abs(effect_size) > 0.8 else 'Medium' if abs(effect_size) > 0.5 else 'Small'
            })

# Save Gap A
results_df = pd.DataFrame(all_results)
results_df.to_csv(f"{STATS_PATH}/Corrected_Statistical_Tests_Effect_Sizes.csv", index=False)

print(f"\nBonferroni correction: α = 0.05/{n_comparisons} = {alpha/n_comparisons:.5f}")
sig_df = results_df[results_df['Significant_after_correction']]
print(f"Significant comparisons: {len(sig_df)}/{len(results_df)}\n")
print("✅ GAP A COMPLETE\n")

# ============================================================
# GAP B: CROSS-FOLD VARIANCE
# ============================================================

print("="*80)
print("GAP B: CROSS-FOLD VARIANCE ANALYSIS")
print("Estimated time: 2 minutes")
print("="*80 + "\n")

fold_variance_results = []

for name in model_names:
    fold_accs = sd_predictions[name]['fold_accs']
    fold_variance_results.append({
        'Model': name,
        'Validation': 'Subject-Dependent',
        'Mean_Accuracy_%': np.mean(fold_accs) * 100,
        'Std_Deviation_%': np.std(fold_accs) * 100,
        'Coefficient_Variation_%': (np.std(fold_accs) / np.mean(fold_accs)) * 100,
        'Min_Fold_%': np.min(fold_accs) * 100,
        'Max_Fold_%': np.max(fold_accs) * 100,
        'Range_pp': (np.max(fold_accs) - np.min(fold_accs)) * 100
    })

for name in si_names:
    fold_accs = si_predictions[name]['fold_accs']
    fold_variance_results.append({
        'Model': name,
        'Validation': 'Subject-Independent',
        'Mean_Accuracy_%': np.mean(fold_accs) * 100,
        'Std_Deviation_%': np.std(fold_accs) * 100,
        'Coefficient_Variation_%': (np.std(fold_accs) / np.mean(fold_accs)) * 100,
        'Min_Fold_%': np.min(fold_accs) * 100,
        'Max_Fold_%': np.max(fold_accs) * 100,
        'Range_pp': (np.max(fold_accs) - np.min(fold_accs)) * 100
    })

fold_var_df = pd.DataFrame(fold_variance_results)
fold_var_df.to_csv(f"{STATS_PATH}/CrossFold_Variance_Analysis.csv", index=False)

print("✅ GAP B COMPLETE\n")

# ============================================================
# GAP C: PER-SUBJECT HEATMAP
# ============================================================

print("="*80)
print("GAP C: PER-SUBJECT ACCURACY HEATMAP")
print("Estimated time: 20 minutes")
print("="*80 + "\n")

# Reload with subject IDs
print("Reloading data with subject tracking...")
X1, y1, subj1 = load_with_subjects(f"{BASE_DATA_PATH}/Relax", 0)
X2, y2, subj2 = load_with_subjects(f"{BASE_DATA_PATH}/Stroop", 1)

X = np.vstack((X1, X2))
y = np.hstack((y1, y2))
all_subjects = np.hstack((subj1, subj2 + 40))

X_scaled = StandardScaler().fit_transform(X)

models_for_heatmap = {
    'KNN (SD)': (KNeighborsClassifier(n_neighbors=5, n_jobs=2), StratifiedKFold(n_splits=5, shuffle=True, random_state=42), False),
    'AdaBoost (SI)': (AdaBoostClassifier(n_estimators=100, random_state=42), GroupKFold(n_splits=5), True),
    'RandomForest (SD)': (RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=2), StratifiedKFold(n_splits=5, shuffle=True, random_state=42), False)
}

per_subject_matrix = []

for model_name, (model, cv, use_groups) in models_for_heatmap.items():
    print(f"Computing per-subject accuracy: {model_name}")

    all_preds = np.zeros(len(y))

    if use_groups:
        groups_for_cv = np.hstack((subj1, subj2 + 40))
        splits = cv.split(X_scaled, y, groups_for_cv)
    else:
        splits = cv.split(X_scaled, y)

    for train_idx, test_idx in splits:
        model_clone = type(model)(**model.get_params())
        model_clone.fit(X_scaled[train_idx], y[train_idx])
        all_preds[test_idx] = model_clone.predict(X_scaled[test_idx])

    unique_subjects = np.unique(all_subjects)
    per_subj_acc = []
    for subj in unique_subjects:
        subj_mask = (all_subjects == subj)
        if np.sum(subj_mask) > 0:
            acc = np.mean(all_preds[subj_mask] == y[subj_mask])
            per_subj_acc.append(acc)

    per_subject_matrix.append(per_subj_acc)

per_subject_matrix = np.array(per_subject_matrix)

# Generate heatmap
print("Generating heatmap...")
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(per_subject_matrix, annot=False, cmap='RdYlGn', vmin=0.3, vmax=1.0,
            xticklabels=[f'S{i}' for i in range(80)],
            yticklabels=list(models_for_heatmap.keys()),
            cbar_kws={'label': 'Accuracy'})
ax.set_xlabel('Subject ID', fontsize=11)
ax.set_title('Per-Subject Accuracy Heatmap (Red=Hard, Green=Easy)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{STATS_PATH}/PerSubject_Accuracy_Heatmap.png", dpi=300, bbox_inches='tight')
plt.close()

# Identify hard subjects
hard_subjects = []
for subj_idx in range(80):
    avg_acc = np.mean(per_subject_matrix[:, subj_idx])
    if avg_acc < 0.45:
        hard_subjects.append(subj_idx)

print(f"Hard subjects identified: {len(hard_subjects)} (accuracy <45%)")

# Save matrix
per_subj_df = pd.DataFrame(per_subject_matrix.T,
                            columns=list(models_for_heatmap.keys()),
                            index=[f'Subject_{i}' for i in range(80)])
per_subj_df.to_csv(f"{STATS_PATH}/PerSubject_Accuracy_Matrix.csv")

print("✅ GAP C COMPLETE\n")

del X, y, all_subjects, X_scaled
gc.collect()

# ============================================================
# GAP D: VERIFIED BOOTSTRAP CIs
# ============================================================

print("="*80)
print("GAP D: VERIFIED BOOTSTRAP CIs (10,000 iterations)")
print("Estimated time: 5 minutes")
print("="*80 + "\n")

ci_results_verified = []

print("Computing bootstrap CIs with 10,000 iterations...")

for name in ['KNN', 'RandomForest']:
    print(f"  {name}...", end=" ")
    fold_accs = sd_predictions[name]['fold_accs']
    mean_acc = np.mean(fold_accs)
    ci_lower, ci_upper = robust_bootstrap_ci(fold_accs, n_bootstrap=10000)

    ci_results_verified.append({
        'Model': name,
        'Validation': 'Subject-Dependent',
        'Mean_Accuracy_%': mean_acc * 100,
        'CI_Lower_%': ci_lower * 100,
        'CI_Upper_%': ci_upper * 100,
        'CI_Width_pp': (ci_upper - ci_lower) * 100,
        'Bootstrap_Iterations': 10000
    })
    print(f"{mean_acc*100:.2f}% [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")

for name in ['AdaBoost', 'CatBoost']:
    print(f"  {name}...", end=" ")
    fold_accs = si_predictions[name]['fold_accs']
    mean_acc = np.mean(fold_accs)
    ci_lower, ci_upper = robust_bootstrap_ci(fold_accs, n_bootstrap=10000)

    ci_results_verified.append({
        'Model': name,
        'Validation': 'Subject-Independent',
        'Mean_Accuracy_%': mean_acc * 100,
        'CI_Lower_%': ci_lower * 100,
        'CI_Upper_%': ci_upper * 100,
        'CI_Width_pp': (ci_upper - ci_lower) * 100,
        'Bootstrap_Iterations': 10000
    })
    print(f"{mean_acc*100:.2f}% [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")

ci_verified_df = pd.DataFrame(ci_results_verified)
ci_verified_df.to_csv(f"{STATS_PATH}/Verified_Bootstrap_CIs_10k.csv", index=False)

print("\n✅ GAP D COMPLETE\n")

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*80)
print("🎉 ALL STATISTICAL GAPS COMPLETED")
print("="*80 + "\n")

print("Generated Files:")
print(f"  1. Corrected_Statistical_Tests_Effect_Sizes.csv")
print(f"  2. CrossFold_Variance_Analysis.csv")
print(f"  3. PerSubject_Accuracy_Heatmap.png")
print(f"  4. PerSubject_Accuracy_Matrix.csv")
print(f"  5. Verified_Bootstrap_CIs_10k.csv")

print("\n" + "="*80)
print("✅ NO MORE CODING REQUIRED")
print("✅ ALL ANALYSIS COMPLETE")
print("✅ FILES SAVED TO GOOGLE DRIVE")
print("="*80)

print(f"\nCompleted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n🎯 Next: Share these files with your partner for paper writing")


STEP 1: Installing packages...

✅ All libraries imported

STEP 2: Mounting Google Drive...

Mounted at /content/drive
✅ Drive mounted
📁 Dataset: /content/drive/MyDrive/Kaggle Dataset
📁 Results: /content/drive/MyDrive/Kaggle Dataset/Results/ResultsAfterOptimization/Statistical_Analysis

✅ Relax folder found
✅ Stroop folder found

STEP 3: Defining helper functions...

✅ Functions defined

STEP 4: Loading EEG data...

✅ Loaded 768,000 samples, 32 features

STEP 5: Defining ML models...

✅ Models defined


GAP A: CORRECTED STATISTICAL TESTS + EFFECT SIZES
Estimated time: 30 minutes

Training Subject-Dependent models (5 models)...
  KNN... Done (Acc: 97.56%)
  RandomForest... 